In [2]:
import numpy as np
import matplotlib.pyplot as plt
import os
join = os.path.join
from tqdm import tqdm
import torch
import torchvision
from torch.utils.data import Dataset, DataLoader
import monai
from segment_anything import SamPredictor, sam_model_registry
from segment_anything.utils.transforms import ResizeLongestSide
import skimage
# MOI ajout des imports qui sont partout dans le doc ici : 
import tarfile
import nibabel as nib
import glob
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact
import seaborn as sns
import pickle
import pandas as pd # Ajouté pour les stats
import fonctions as fc

In [12]:
import importlib
import fonctions as fc
importlib.reload(fc)

<module 'fonctions' from '/home/cassa2/psy3019/projet/codes_reproductibles/fonctions.py'>

In [3]:
with open("results.pkl", "rb") as f:
    results_load = pickle.load(f)

print(list(results_load.keys()))  # les sujets
print(list(next(iter(results_load.values())).keys()))  # les variables

['BraTS2021_00000', 'BraTS2021_00002']
['slice_index', 'image', 'gt', 'sam_seg', 'medsam_seg']


In [15]:
# Make initial table 

tableau_resultat = pd.DataFrame(index=results_load.keys())
tableau_resultat["slice_index"] = [results_load[s]["slice_index"] for s in results_load]
print(tableau_resultat)

                 slice_index
BraTS2021_00000           80
BraTS2021_00002           80


In [16]:
for sujet in tableau_resultat.index:

    sam_seg = results_load[sujet]['sam_seg']
    medsam_seg = results_load[sujet]['medsam_seg']
    gt = results_load[sujet]['gt']

    # Ajout colone dice score 
    sam_dsc = fc.compute_dice_coefficient(gt > 0, sam_seg > 0)
    medsam_dsc = fc.compute_dice_coefficient(gt > 0, medsam_seg > 0)

    # Ajout colone precision, recall 
    precision_sam, recall_sam = fc.precision_recall(gt > 0, sam_seg > 0)
    precision_medsam, recall_medsam = fc.precision_recall(gt > 0, medsam_seg > 0)

    tableau_resultat.loc[sujet, "sam_dice"] = sam_dsc
    tableau_resultat.loc[sujet, "sam_precision"] = precision_sam
    tableau_resultat.loc[sujet, "sam_recall"] = recall_sam
    tableau_resultat.loc[sujet, "medsam_dice"] = medsam_dsc
    tableau_resultat.loc[sujet, "medsam_precision"] = precision_medsam
    tableau_resultat.loc[sujet, "medsam_recall"] = recall_medsam

print(tableau_resultat)
print(tableau_resultat.columns)

                 slice_index  sam_dice  sam_precision  sam_recall  \
BraTS2021_00000           80  0.716989       0.568902    0.969301   
BraTS2021_00002           80  0.749437       0.605593    0.982903   

                 medsam_dice  medsam_precision  medsam_recall  
BraTS2021_00000     0.863097          0.879127       0.847641  
BraTS2021_00002     0.810496          0.879857       0.751272  
Index(['slice_index', 'sam_dice', 'sam_precision', 'sam_recall', 'medsam_dice',
       'medsam_precision', 'medsam_recall'],
      dtype='str')


Dice : Quantifie le chevauchement 
Précision : Si il déborde  
Recall : Si il manque des éléments de la tumeur 

In [ ]:
sujet = 'BraTS2021_00000'

sam_seg = results_load[sujet]['sam_seg']
medsam_seg = results_load[sujet]['medsam_seg']
gt = results_load[sujet]['gt']




print(precision_medsam)

0.8476407049459921
